# Pratica — Lezione 10. Rainbow e apprendimento distribuzionale

**DIH DATalife — Corso di Reinforcement Learning (RL) da zero**

In questa pratica programmi tu stesso due componenti di Rainbow a partire dal DQN della lezione sette:

1. L'architettura dueling completa: in classe l'hai vista già montata, oggi la costruisci da zero
2. I ritorni multi-step: il miglioramento che lo studio di ablazione ha indicato come uno dei più utili per il rendimento finale

Usa lo stesso ambiente CartPole-v1 e lo stesso ciclo di addestramento della lezione.

**Prima di iniziare, esegui le celle della sezione "Preparazione".** Non serve modificarle.

## Preparazione

Installa gymnasium e definisce i componenti che già conosci dalla lezione: il buffer di replay, la rete del DQN standard (che funge da riferimento) e il ciclo di addestramento completo.

In [ ]:
!pip install gymnasium


In [ ]:
import random
from collections import deque, namedtuple

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Fijamos semillas para que el experimento sea reproducible
SEMILLA = 0
random.seed(SEMILLA)
np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)


In [ ]:
Transicion = namedtuple(
    "Transicion", ["estado", "accion", "recompensa", "siguiente_estado", "terminado"]
)


class BufferReplay:
    """Memoria de repeticion de experiencia, identica a la de la clase."""

    def __init__(self, capacidad=10000):
        self.memoria = deque(maxlen=capacidad)

    def guardar(self, *args):
        self.memoria.append(Transicion(*args))

    def muestrear(self, tamano_lote):
        return random.sample(self.memoria, tamano_lote)

    def __len__(self):
        return len(self.memoria)


class RedQEstandar(nn.Module):
    """DQN "de manual": una sola cabeza que produce directamente
    un valor Q por cada accion posible. La usas como linea base
    en el ejercicio uno y como red interna del ejercicio dos."""

    def __init__(self, n_entradas, n_acciones, n_ocultas=128):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_entradas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_acciones),
        )

    def forward(self, x):
        return self.red(x)


def entrenar(clase_red, nombre, n_episodios=100, gamma=0.99, tamano_lote=64,
             tasa_aprendizaje=1e-3, actualizar_objetivo_cada=10):
    """Bucle de entrenamiento DQN completo, identico al de la clase."""
    entorno = gym.make("CartPole-v1")
    n_entradas = entorno.observation_space.shape[0]
    n_acciones = entorno.action_space.n

    red_online = clase_red(n_entradas, n_acciones)
    red_objetivo = clase_red(n_entradas, n_acciones)
    red_objetivo.load_state_dict(red_online.state_dict())
    red_objetivo.eval()

    optimizador = optim.Adam(red_online.parameters(), lr=tasa_aprendizaje)
    buffer = BufferReplay(capacidad=10000)

    epsilon, epsilon_minimo, decaimiento = 1.0, 0.05, 0.97
    recompensas_por_episodio = []

    for episodio in range(n_episodios):
        estado, _ = entorno.reset(seed=SEMILLA + episodio)
        recompensa_total = 0.0
        terminado = False

        while not terminado:
            if random.random() < epsilon:
                accion = entorno.action_space.sample()
            else:
                with torch.no_grad():
                    q_valores = red_online(torch.tensor(estado, dtype=torch.float32))
                    accion = int(torch.argmax(q_valores).item())

            siguiente_estado, recompensa, fin, truncado, _ = entorno.step(accion)
            terminado = fin or truncado
            buffer.guardar(estado, accion, recompensa, siguiente_estado, fin)
            estado = siguiente_estado
            recompensa_total += recompensa

            if len(buffer) >= tamano_lote:
                lote = buffer.muestrear(tamano_lote)
                estados = torch.tensor(np.array([t.estado for t in lote]), dtype=torch.float32)
                acciones = torch.tensor([t.accion for t in lote], dtype=torch.int64)
                recompensas = torch.tensor([t.recompensa for t in lote], dtype=torch.float32)
                siguientes = torch.tensor(np.array([t.siguiente_estado for t in lote]), dtype=torch.float32)
                terminados = torch.tensor([t.terminado for t in lote], dtype=torch.float32)

                q_actuales = red_online(estados).gather(1, acciones.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    q_siguientes = red_objetivo(siguientes).max(dim=1)[0]
                    objetivo = recompensas + gamma * q_siguientes * (1 - terminados)

                perdida = nn.functional.mse_loss(q_actuales, objetivo)
                optimizador.zero_grad()
                perdida.backward()
                optimizador.step()

        recompensas_por_episodio.append(recompensa_total)
        epsilon = max(epsilon_minimo, epsilon * decaimiento)

        if episodio % actualizar_objetivo_cada == 0:
            red_objetivo.load_state_dict(red_online.state_dict())

    entorno.close()
    print(f"{nombre}: recompensa media de los ultimos 20 episodios = "
          f"{np.mean(recompensas_por_episodio[-20:]):.1f}")
    return recompensas_por_episodio


### Addestriamo la linea di base

**Esegui questa cella prima di iniziare gli esercizi.** Userai `recompensas_dqn_estandar` come punto di confronto nell'esercizio uno. Richiede circa mezzo minuto.

In [ ]:
recompensas_dqn_estandar = entrenar(RedQEstandar, "DQN estándar (línea base)")


## Esercizio 1 — Completa l'architettura dueling da zero

In classe hai visto l'architettura dueling già costruita. Ora tocca a te: completa `RedQDuelingIncompleta`, dove c'è `# TODO`.

Ricorda l'idea: un tronco condiviso alimenta due teste.

- La **testa del valore** stima V(s): quanto vale lo stato in generale, un unico numero
- La **testa del vantaggio** stima A(s,a): quanto è migliore ciascuna azione concreta rispetto alle altre, un numero per ogni azione possibile

Combinandole, la formula è:

**Q(s,a) = V(s) + (A(s,a) − media_a[A(s,a)])**

**Suggerimenti**:

1. Ciascuna testa è un piccolo `nn.Sequential`: un livello lineare nascosto seguito da `nn.ReLU()`, e poi un livello lineare finale
2. La testa del valore termina con un solo neurone (`nn.Linear(n_ocultas, 1)`); la testa del vantaggio termina con `n_acciones` neuroni
3. Per sottrarre la media del vantaggio lungo le azioni, usa `ventaja.mean(dim=-1, keepdim=True)`

**Il tuo compito è completare i tre `# TODO`.**

In [ ]:
class RedQDuelingIncompleta(nn.Module):
    """Arquitectura dueling: un tronco compartido alimenta dos cabezas.
    Completa las dos cabezas y la formula de combinacion siguiendo
    las pistas del enunciado."""

    def __init__(self, n_entradas, n_acciones, n_ocultas=128):
        super().__init__()
        self.tronco = nn.Sequential(
            nn.Linear(n_entradas, n_ocultas),
            nn.ReLU(),
        )

        # TODO 1: cabeza de valor V(s) -- debe terminar en una sola neurona
        self.cabeza_valor = None

        # TODO 2: cabeza de ventaja A(s,a) -- debe terminar en n_acciones neuronas
        self.cabeza_ventaja = None

    def forward(self, x):
        caracteristicas = self.tronco(x)
        valor = self.cabeza_valor(caracteristicas)
        ventaja = self.cabeza_ventaja(caracteristicas)

        # TODO 3: combina valor y ventaja: Q(s,a) = V(s) + (A(s,a) - media_a[A(s,a)])
        return None


**Esegui la cella seguente per addestrare il tuo Dueling DQN e confrontarlo con la linea di base.** Richiede circa mezzo minuto.

In [ ]:
recompensas_dueling = entrenar(RedQDuelingIncompleta, "Dueling DQN (tu implementacion)")

plt.figure(figsize=(10, 5))
plt.plot(recompensas_dqn_estandar, label="DQN estandar (linea base)", alpha=0.6)
plt.plot(recompensas_dueling, label="Dueling DQN (tu implementacion)", alpha=0.6)
plt.xlabel("Episodio")
plt.ylabel("Recompensa")
plt.title("CartPole-v1: DQN estandar vs. tu Dueling DQN")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


<details>
<summary>💡 Vedi la soluzione</summary>

```python
class RedQDuelingIncompleta(nn.Module):
    """Architettura dueling: un tronco condiviso alimenta due teste.
    Una stima V(s) -- quanto vale lo stato, un unico numero --
    e l'altra stima A(s,a) -- quanto è migliore ciascuna azione rispetto alle altre."""

    def __init__(self, n_entradas, n_acciones, n_ocultas=128):
        super().__init__()
        self.tronco = nn.Sequential(
            nn.Linear(n_entradas, n_ocultas),
            nn.ReLU(),
        )

        # Testa del valore V(s): termina con un solo neurone
        self.cabeza_valor = nn.Sequential(
            nn.Linear(n_ocultas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, 1),
        )

        # Testa del vantaggio A(s,a): termina con n_acciones neuroni
        self.cabeza_ventaja = nn.Sequential(
            nn.Linear(n_ocultas, n_ocultas),
            nn.ReLU(),
            nn.Linear(n_ocultas, n_acciones),
        )

    def forward(self, x):
        caracteristicas = self.tronco(x)
        valor = self.cabeza_valor(caracteristicas)
        ventaja = self.cabeza_ventaja(caracteristicas)
        # Q(s,a) = V(s) + (A(s,a) - media_a[A(s,a)])
        return valor + (ventaja - ventaja.mean(dim=-1, keepdim=True))
```

In CartPole è normale che il Dueling DQN non superi sempre la linea di base: è un ambiente con pochi stati in cui quasi tutte le azioni influiscono già sul risultato, quindi l'architettura dueling non ha tanto margine per mostrare il proprio valore come in ambienti più grandi.
</details>

## Esercizio 2 — Aggiungi i ritorni multi-step al ciclo di addestramento

Il DQN standard guarda un solo passo in avanti prima di aggiornare le sue stime. Ora accumulerai la ricompensa di più passi consecutivi (n = 3) prima di fare bootstrap con la rete obiettivo, e confronterai la curva risultante con quella del DQN a un solo passo.

Qui sotto trovi `entrenar_nstep_incompleto`, una copia del ciclo di addestramento che già mantiene una finestra `ventana_nstep` con gli ultimi n passi —stato, azione, ricompensa— e che sa già fare bootstrap con `gamma ** n_pasos` una volta disponibile il ritorno accumulato. **Manca solo calcolare quel ritorno.**

**Suggerimenti**:

1. `ventana_nstep` è un `deque` con fino a `n_pasos` transizioni `(estado, accion, recompensa)`, ordinate dalla più vecchia alla più recente
2. Il ritorno multi-step sconta ogni ricompensa in base alla sua posizione nella finestra: la prima senza sconto (gamma elevato a zero), la seconda moltiplicata per gamma, la terza per gamma al quadrato, e così via
3. Scorri `ventana_nstep` con `enumerate` per ottenere contemporaneamente la posizione `i` e la ricompensa `r` di ogni transizione

**Il tuo compito è completare il `# TODO`.**

In [ ]:
def entrenar_nstep_incompleto(nombre, n_pasos=1, n_episodios=100, gamma=0.99, tamano_lote=64,
                               tasa_aprendizaje=1e-3, actualizar_objetivo_cada=10):
    """Bucle de entrenamiento DQN con retornos multi-paso. Con n_pasos=1
    equivale al DQN estandar de la seccion de preparacion."""
    entorno = gym.make("CartPole-v1")
    n_entradas = entorno.observation_space.shape[0]
    n_acciones = entorno.action_space.n

    red_online = RedQEstandar(n_entradas, n_acciones)
    red_objetivo = RedQEstandar(n_entradas, n_acciones)
    red_objetivo.load_state_dict(red_online.state_dict())
    red_objetivo.eval()

    optimizador = optim.Adam(red_online.parameters(), lr=tasa_aprendizaje)
    buffer = BufferReplay(capacidad=10000)

    epsilon, epsilon_minimo, decaimiento = 1.0, 0.05, 0.97
    recompensas_por_episodio = []

    for episodio in range(n_episodios):
        estado, _ = entorno.reset(seed=SEMILLA + episodio)
        recompensa_total = 0.0
        terminado = False
        ventana_nstep = deque(maxlen=n_pasos)  # guarda (estado, accion, recompensa)

        while not terminado:
            if random.random() < epsilon:
                accion = entorno.action_space.sample()
            else:
                with torch.no_grad():
                    q_valores = red_online(torch.tensor(estado, dtype=torch.float32))
                    accion = int(torch.argmax(q_valores).item())

            siguiente_estado, recompensa, fin, truncado, _ = entorno.step(accion)
            terminado = fin or truncado
            ventana_nstep.append((estado, accion, recompensa))
            estado = siguiente_estado
            recompensa_total += recompensa

            # En marcha normal emitimos una transicion n-step por paso; al terminar
            # el episodio vaciamos toda la ventana que quede pendiente.
            while len(ventana_nstep) == n_pasos or (terminado and len(ventana_nstep) > 0):
                estado_inicial, accion_inicial, _ = ventana_nstep[0]

                # TODO: calcula el retorno acumulado de las recompensas que quedan en
                # `ventana_nstep`, descontando cada una segun su posicion (gamma**i),
                # antes de que el bootstrap de mas abajo aplique el descuento final
                # gamma**n_pasos sobre el valor del estado siguiente
                retorno_nstep = None

                buffer.guardar(
                    estado_inicial, accion_inicial, retorno_nstep,
                    siguiente_estado, fin,
                )
                ventana_nstep.popleft()

            if len(buffer) >= tamano_lote:
                lote = buffer.muestrear(tamano_lote)
                estados = torch.tensor(np.array([t.estado for t in lote]), dtype=torch.float32)
                acciones = torch.tensor([t.accion for t in lote], dtype=torch.int64)
                recompensas = torch.tensor([t.recompensa for t in lote], dtype=torch.float32)
                siguientes = torch.tensor(np.array([t.siguiente_estado for t in lote]), dtype=torch.float32)
                terminados = torch.tensor([t.terminado for t in lote], dtype=torch.float32)

                q_actuales = red_online(estados).gather(1, acciones.unsqueeze(1)).squeeze(1)
                with torch.no_grad():
                    q_siguientes = red_objetivo(siguientes).max(dim=1)[0]
                    objetivo = recompensas + (gamma ** n_pasos) * q_siguientes * (1 - terminados)

                perdida = nn.functional.mse_loss(q_actuales, objetivo)
                optimizador.zero_grad()
                perdida.backward()
                optimizador.step()

        recompensas_por_episodio.append(recompensa_total)
        epsilon = max(epsilon_minimo, epsilon * decaimiento)

        if episodio % actualizar_objetivo_cada == 0:
            red_objetivo.load_state_dict(red_online.state_dict())

    entorno.close()
    print(f"{nombre}: recompensa media de los ultimos 20 episodios = "
          f"{np.mean(recompensas_por_episodio[-20:]):.1f}")
    return recompensas_por_episodio


**Esegui la cella seguente per confrontare un passo con tre passi.** Richiede circa un minuto in totale.

In [ ]:
recompensas_1paso = entrenar_nstep_incompleto("DQN 1 paso", n_pasos=1)
recompensas_3pasos = entrenar_nstep_incompleto("DQN 3 pasos", n_pasos=3)

plt.figure(figsize=(10, 5))
plt.plot(recompensas_1paso, label="DQN 1 paso", alpha=0.6)
plt.plot(recompensas_3pasos, label="DQN 3 pasos", alpha=0.6)
plt.xlabel("Episodio")
plt.ylabel("Recompensa")
plt.title("CartPole-v1: retornos de un paso vs. retornos multi-paso (n=3)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


<details>
<summary>💡 Vedi la soluzione</summary>

```python
            while len(ventana_nstep) == n_pasos or (terminado and len(ventana_nstep) > 0):
                estado_inicial, accion_inicial, _ = ventana_nstep[0]

                # Sommiamo ogni ricompensa della finestra scontata in base alla sua posizione:
                # la più vecchia (i=0) senza sconto, la successiva per gamma, ecc.
                retorno_nstep = 0.0
                for i, (_, _, r) in enumerate(ventana_nstep):
                    retorno_nstep += (gamma ** i) * r

                buffer.guardar(
                    estado_inicial, accion_inicial, retorno_nstep,
                    siguiente_estado, fin,
                )
                ventana_nstep.popleft()
```

Con `n_pasos=3` l'agente di solito impara più in fretta che con un solo passo: ogni aggiornamento incorpora tre ricompense reali invece di una, quindi il segnale di apprendimento arriva con meno rumore e il bootstrap con la rete obiettivo pesa meno sul risultato.
</details>

## Per continuare a esercitarti

**Porta `n_pasos` a cinque nell'esercizio due e riesegui il confronto.** La curva continua a migliorare quanto più lontano guarda l'agente, o a un certo punto comincia a peggiorare?

Condividi il risultato con il tuo team e **confrontate** quale combinazione di miglioramenti ha prodotto la curva di apprendimento più stabile.